# 01 · Bronze · Olist (canal marketplace)
Ingere os CSVs do dataset público Olist, que representa o canal **marketplace** da Lojas Horizonte.

- **Origem:** `/Volumes/horizonte/bronze/landing/olist/`
- **Fonte:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), licença CC BY-NC-SA 4.0
- **Tabelas geradas:** `olist_orders`, `olist_order_items`, `olist_products`, `olist_customers`, `olist_category_translation`

### Ingestão
Decisões de bronze:
- **Tudo é lido como texto** (`inferSchema=False`): a bronze guarda o dado exatamente como chegou. A tipagem acontece na silver.
- **Colunas de auditoria:** `_arquivo_origem` e `_ingerido_em` registram de onde e quando cada linha veio, base da linhagem.
- **Modo `overwrite`:** a carga é completa a cada execução, adequado a um dataset estático.

Contagens de referência: 99.441 pedidos · 112.650 itens · 32.951 produtos · 99.441 clientes · 71 traduções de categoria.

In [0]:
from pyspark.sql import functions as F

CATALOGO = "horizonte"   
ORIGEM = f"/Volumes/{CATALOGO}/bronze/landing/olist"

arquivos = {
    "olist_orders": "olist_orders_dataset.csv",
    "olist_order_items": "olist_order_items_dataset.csv",
    "olist_products": "olist_products_dataset.csv",
    "olist_customers": "olist_customers_dataset.csv",
    "olist_category_translation": "product_category_name_translation.csv",
}

for tabela, arquivo in arquivos.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)     #guarda tudo como texto, como veio
          .csv(f"{ORIGEM}/{arquivo}")
          .withColumn("_arquivo_origem", F.lit(arquivo))    #registra onde cada linha veio
          .withColumn("_ingerido_em", F.current_timestamp()))   #registra quando cada linha veio
    df.write.mode("overwrite").saveAsTable(f"{CATALOGO}.bronze.{tabela}")
    print(f"{tabela}: {df.count():,} linhas")